# 00 — OECD data audit and cleaning

This notebook cleans the OECD Current Well-being dataset before I use it in the rest of the project. The specific steps are as follows: check the source file, clean up the columns and rows, and finally save four CSV files for further analysis.

**Input:** `data/raw/OECD Data.csv`

**Outputs:** `data/processed/OECD_cleaned_version.csv`, `data/processed/OECD_categories.csv`, `data/processed/OECD_domain_counts.csv`, and `data/processed/AU_domain_counts.csv`


## 1. Setup and file locations

Set the paths at the start so the notebook can run from either the project folder or the `notebooks` folder.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_FILE = PROJECT_ROOT / 'data' / 'raw' / 'OECD Data.csv'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TIDY_FILE = PROCESSED_DIR / 'OECD_cleaned_version.csv'
SUMMARY_FILE = PROCESSED_DIR / 'OECD_categories.csv'
COUNTS_FILE = PROCESSED_DIR / 'OECD_domain_counts.csv'
AU_COUNTS_FILE = PROCESSED_DIR / 'AU_domain_counts.csv'

if not RAW_FILE.exists():
    raise FileNotFoundError(f'Raw input not found: {RAW_FILE}')

RAW_FILE

PosixPath('/Users/y2l/winter-data-challenge/data/raw/OECD Data.csv')

## 2. Load the raw data

Load the original dataset and use the first five rows to test whether it was successful.

In [2]:
raw = pd.read_csv(RAW_FILE, low_memory=False)
print(f'Raw shape: {raw.shape[0]:,} rows × {raw.shape[1]:,} columns')
display(raw.head())

Raw shape: 8,806 rows × 30 columns


,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,ACTION,REF_AREA,Reference area,MEASURE,Measure,UNIT_MEASURE,Unit of measure,...,OBS_VALUE,Observation value,OBS_STATUS,Observation status,UNIT_MULT,Unit multiplier,DECIMALS,Decimals,BASE_PER,Base period
0,DATAFLOW,OECD.WISE.WDP:DSD_HSL@DF_HSL_CWB(1.1),Current well-being,I,AUS,Australia,1_1,Households and NPISHs net adjusted disposable ...,USD_PPP_PS,"US dollars per person, PPP converted",...,44625.0,NaN,A,Normal value,0,Units,2,Two,2023,NaN
1,DATAFLOW,OECD.WISE.WDP:DSD_HSL@DF_HSL_CWB(1.1),Current well-being,I,AUS,Australia,1_1,Households and NPISHs net adjusted disposable ...,USD_PPP_PS,"US dollars per person, PPP converted",...,45369.0,NaN,A,Normal value,0,Units,2,Two,2023,NaN
2,DATAFLOW,OECD.WISE.WDP:DSD_HSL@DF_HSL_CWB(1.1),Current well-being,I,AUS,Australia,1_1,Households and NPISHs net adjusted disposable ...,USD_PPP_PS,"US dollars per person, PPP converted",...,44830.0,NaN,A,Normal value,0,Units,2,Two,2023,NaN
3,DATAFLOW,OECD.WISE.WDP:DSD_HSL@DF_HSL_CWB(1.1),Current well-being,I,AUS,Australia,1_1,Households and NPISHs net adjusted disposable ...,USD_PPP_PS,"US dollars per person, PPP converted",...,45377.0,NaN,A,Normal value,0,Units,2,Two,2023,NaN
4,DATAFLOW,OECD.WISE.WDP:DSD_HSL@DF_HSL_CWB(1.1),Current well-being,I,AUS,Australia,1_1,Households and NPISHs net adjusted disposable ...,USD_PPP_PS,"US dollars per person, PPP converted",...,46031.0,NaN,A,Normal value,0,Units,2,Two,2023,NaN


## 3. Audit columns, missingness, and duplicates

This audit distinguishes between actual missing data and blank values in descriptive export columns.

In [3]:
audit = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'missing_count': raw.isna().sum(),
    'missing_percent': raw.isna().mean().mul(100).round(2),
    'unique_non_missing': raw.nunique(dropna=True),
}).sort_values(['missing_percent', 'unique_non_missing'], ascending=[False, True])
display(audit)

key_raw = ['REF_AREA', 'MEASURE', 'TIME_PERIOD']
print('Exact duplicate rows:', int(raw.duplicated().sum()))
print('Duplicate country-indicator-year keys:', int(raw.duplicated(key_raw).sum()))

,dtype,missing_count,missing_percent,unique_non_missing
Time period,float64,8806,100.0,0
Observation value,float64,8806,100.0,0
Base period,float64,8806,100.0,0
STRUCTURE,object,0,0.0,1
STRUCTURE_ID,object,0,0.0,1
STRUCTURE_NAME,object,0,0.0,1
ACTION,object,0,0.0,1
AGE,object,0,0.0,1
Age,object,0,0.0,1
SEX,object,0,0.0,1


Exact duplicate rows: 0
Duplicate country-indicator-year keys: 0


## 4. Define cleaning rules

Remove some useless information in the orignial dataset:

- `Time period`, `Observation value` and `Base period` are completely empty;
- `STRUCTURE`, `STRUCTURE_ID`, `STRUCTURE_NAME` and `ACTION` contain only one value;
- the age, sex and education columns are all set to total;
- the multiplier and decimal columns are also constant.

The nation, indication, domain, unit, and observation status codes and human-readable labels are kept. Non-standard OECD status records are kept with a flag because they are still useful.

In [4]:
drop_columns = [
    'STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION',
    'AGE', 'Age', 'SEX', 'Sex', 'EDUCATION_LEV', 'Education level',
    'Time period', 'Observation value', 'Base period',
    'UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals',
]

rename_columns = {
    'REF_AREA': 'country_code',
    'Reference area': 'country',
    'MEASURE': 'indicator_code',
    'Measure': 'indicator',
    'UNIT_MEASURE': 'unit_code',
    'Unit of measure': 'unit',
    'DOMAIN': 'domain_code',
    'Domain': 'domain',
    'TIME_PERIOD': 'year',
    'OBS_VALUE': 'value',
    'OBS_STATUS': 'status_code',
    'Observation status': 'status',
    'BASE_PER': 'base_period',
}

df = raw.drop(columns=drop_columns).rename(columns=rename_columns).copy()
df.columns.tolist()

['country_code',
 'country',
 'indicator_code',
 'indicator',
 'unit_code',
 'unit',
 'domain_code',
 'domain',
 'year',
 'value',
 'status_code',
 'status',
 'base_period']

## 5. Standardise text and correct data types

Remove extra spaces from the text columns and convert year and value to numeric types. 

In the base-period column, `_Z` (not applicable rather than a real base year) has been replaced to `NA`.

In [5]:
text_columns = [
    'country_code', 'country', 'indicator_code', 'indicator',
    'unit_code', 'unit', 'domain_code', 'domain',
    'status_code', 'status', 'base_period',
]
for column in text_columns:
    df[column] = df[column].astype('string').str.strip()

df['base_period'] = df['base_period'].replace({'_Z': pd.NA, '': pd.NA})
df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
df['value'] = pd.to_numeric(df['value'], errors='coerce').astype('Float64')
df.dtypes

country_code      string[python]
country           string[python]
indicator_code    string[python]
indicator         string[python]
unit_code         string[python]
unit              string[python]
domain_code       string[python]
domain            string[python]
year                       Int64
value                    Float64
status_code       string[python]
status            string[python]
base_period       string[python]
dtype: object

## 6. Remove invalid and out-of-scope records

A valid data row must include country, indicator, domain, unit, year, and observed value. If any of these fields is missing, the row will be deleted. Additionally, data from 2010–2024 has been retained, as this is the time period used in this challenge. Duplicate data has also been filtered out.

In [6]:
required_columns = [
    'country_code', 'country', 'indicator_code', 'indicator',
    'unit_code', 'unit', 'domain_code', 'domain', 'year', 'value',
]
invalid_core_mask = df[required_columns].isna().any(axis=1)
outside_period_mask = ~df['year'].between(2010, 2024, inclusive='both')

cleaning_log = pd.Series({
    'raw_rows': len(df),
    'invalid_core_rows_removed': int(invalid_core_mask.sum()),
    'outside_2010_2024_removed': int((outside_period_mask & ~invalid_core_mask).sum()),
}, name='count')

df = df.loc[~invalid_core_mask & ~outside_period_mask].copy()
exact_duplicates = int(df.duplicated().sum())
df = df.drop_duplicates()

observation_key = ['country_code', 'indicator_code', 'year']
key_duplicates = int(df.duplicated(observation_key).sum())
df = df.drop_duplicates(observation_key, keep='first')

cleaning_log['exact_duplicates_removed'] = exact_duplicates
cleaning_log['duplicate_keys_removed'] = key_duplicates
cleaning_log['tidy_rows'] = len(df)
display(cleaning_log.to_frame())

,count
raw_rows,8806
invalid_core_rows_removed,0
outside_2010_2024_removed,231
exact_duplicates_removed,0
duplicate_keys_removed,0
tidy_rows,8575


## 7. Add data-quality flags

Most records are marked as `Normal value`. The estimated and provisional values are kept with a definition difference or time-series break. The two new Boolean columns are added for the later checks.

In [7]:
df['is_normal_value'] = df['status_code'].eq('A')
df['has_quality_caveat'] = ~df['is_normal_value']
display(df.groupby(['status_code', 'status'], dropna=False).size().rename('row_count').reset_index())

,status_code,status,row_count
0,A,Normal value,8321
1,B,Time series break,39
2,D,Definition differs,45
3,E,Estimated value,85
4,P,Provisional value,85


## 8. Validate the tidy structure

Each row should be one country, one indicator and one year. A country code should have one country name, and each indicator should have one name, domain and unit in this dataset. If any of these checks fail, the code stops instead of exporting an incorrect file.

In [8]:
assert not df.empty
assert df[required_columns].notna().all().all()
assert df['year'].between(2010, 2024).all()
assert not df.duplicated(observation_key).any()
assert df.groupby('country_code')['country'].nunique().max() == 1
assert df.groupby('indicator_code')['indicator'].nunique().max() == 1
assert df.groupby('indicator_code')['domain_code'].nunique().max() == 1
assert df.groupby('indicator_code')['unit_code'].nunique().max() == 1

missing_check = pd.DataFrame({
    'missing_count': df.isna().sum(),
    'missing_percent': df.isna().mean().mul(100).round(2),
})
display(missing_check)
print(f'Validated tidy shape: {len(df):,} rows × {df.shape[1]:,} columns')

,missing_count,missing_percent
country_code,0,0.00
country,0,0.00
indicator_code,0,0.00
indicator,0,0.00
unit_code,0,0.00
unit,0,0.00
domain_code,0,0.00
domain,0,0.00
year,0,0.00
value,0,0.00


Validated tidy shape: 8,575 rows × 15 columns


## 9. Order rows and export the tidy dataset

The columns are arranged in a practical order and sort the rows by domain, indicator, country and year. The result stays in long format, with one observation per row.

In [9]:
column_order = [
    'country_code', 'country', 'indicator_code', 'indicator',
    'domain_code', 'domain', 'unit_code', 'unit', 'year', 'value',
    'status_code', 'status', 'is_normal_value', 'has_quality_caveat', 'base_period',
]
tidy = (df[column_order]
        .sort_values(['domain', 'indicator', 'country', 'year'], kind='stable')
        .reset_index(drop=True))
tidy.to_csv(TIDY_FILE, index=False)
display(tidy.head())
print(f'Saved {TIDY_FILE} ({len(tidy):,} rows)')

,country_code,country,indicator_code,indicator,domain_code,domain,unit_code,unit,year,value,status_code,status,is_normal_value,has_quality_caveat,base_period
0,AUS,Australia,8_1_DEP,Not having a say in government,HSL_8,Civic engagement,PT_POP_Y16T65,Percentage of population aged 16-65 years,2021,46.815718,A,Normal value,True,False,<NA>
1,AUS,Australia,8_1_DEP,Not having a say in government,HSL_8,Civic engagement,PT_POP_Y16T65,Percentage of population aged 16-65 years,2023,35.585077,A,Normal value,True,False,<NA>
2,AUT,Austria,8_1_DEP,Not having a say in government,HSL_8,Civic engagement,PT_POP_Y16T65,Percentage of population aged 16-65 years,2021,63.999894,A,Normal value,True,False,<NA>
3,BEL,Belgium,8_1_DEP,Not having a say in government,HSL_8,Civic engagement,PT_POP_Y16T65,Percentage of population aged 16-65 years,2021,58.966369,A,Normal value,True,False,<NA>
4,BEL,Belgium,8_1_DEP,Not having a say in government,HSL_8,Civic engagement,PT_POP_Y16T65,Percentage of population aged 16-65 years,2023,53.506014,A,Normal value,True,False,<NA>


Saved /Users/y2l/winter-data-challenge/data/processed/OECD_cleaned_version.csv (8,575 rows)


## 10. Create the domain/category summary

The OECD dataset uses the word **domain** for a category. One summary row is made for each domain and list its indicators, indicator codes and units. Also it includes the number of indicators, observations and countries, plus the first and last available years. The ` | ` symbol separates multiple items inside one CSV cell.

In [10]:
def join_unique(values):
    return ' | '.join(sorted(pd.Series(values).dropna().astype(str).unique()))

domain_summary = (
    tidy.groupby(['domain_code', 'domain'], as_index=False)
        .agg(
            indicators=('indicator', join_unique),
            indicator_codes=('indicator_code', join_unique),
            units=('unit', join_unique),
            indicator_count=('indicator_code', 'nunique'),
            observation_count=('value', 'count'),
            country_count=('country_code', 'nunique'),
            first_year=('year', 'min'),
            last_year=('year', 'max'),
        )
        .sort_values('domain')
        .reset_index(drop=True)
)
domain_summary.to_csv(SUMMARY_FILE, index=False)
display(domain_summary)
print(f'Saved {SUMMARY_FILE}')

,domain_code,domain,indicators,indicator_codes,units,indicator_count,observation_count,country_count,first_year,last_year
0,HSL_8,Civic engagement,Not having a say in government | Voter turnout,8_1_DEP | 8_2,Percentage of population aged 16-65 years | Pe...,2,247,47,2010,2024
1,HSL_9,Environmental quality,Exposed to air pollution | Exposure to extreme...,9_2 | 9_3,Percentage of population,2,1211,47,2010,2024
2,HSL_5,Health,"Deaths from suicide, alcohol, drugs | Life exp...",5_1 | 5_3,Deaths per 100 000 inhabitants | Years,2,1283,47,2010,2024
3,HSL_3,Housing,Households living in overcrowded conditions | ...,3_1 | 3_2,Percentage of household gross adjusted disposa...,2,1062,40,2010,2024
4,HSL_1,Income and wealth,Households and NPISHs net adjusted disposable ...,1_1 | 1_2 | 1_3,Factor of lowest income quintile | US dollars ...,3,1138,42,2010,2024
5,HSL_6,Knowledge and skills,Student mathematics skills,6_2,Points,1,154,39,2012,2022
6,HSL_7,Social connections,Lack of social support | Time spent in social ...,7_1_DEP | 7_2,Hours per week | Percentage of population aged...,2,728,47,2010,2024
7,HSL_11,Subjective well-being,Life satisfaction | Negative affect balance,11_1 | 11_2,0-10 scale | Percentage of population aged 15 ...,2,929,47,2010,2024
8,HSL_2,Work and job quality,Employment rate | Gender wage gap | Long hours...,2_1 | 2_2 | 2_7,Percentage of employees | Percentage of popula...,3,1772,47,2010,2024
9,HSL_4,Work-life balance,Gender gap in working hours | Time off,4_1 | 4_3,Hours per day | Minutes per day,2,51,16,2010,2022


Saved /Users/y2l/winter-data-challenge/data/processed/OECD_categories.csv


## 11. Rank domains from highest to lowest data count

Since this is a long-format dataset, the fields are sorted by the number of rows containing observations rather than by the number of columns. The number of indicators and countries is still included to illustrate the composition behind the total figures.

In [11]:
domain_counts = domain_summary[[
    'domain_code', 'domain', 'observation_count', 'indicator_count',
    'country_count', 'first_year', 'last_year',
]].copy()
domain_counts['share_of_all_observations_pct'] = (
    domain_counts['observation_count'].div(len(tidy)).mul(100).round(2)
)
domain_counts = (domain_counts
                 .sort_values(['observation_count', 'domain'], ascending=[False, True])
                 .reset_index(drop=True))
domain_counts.insert(0, 'rank', domain_counts.index + 1)
domain_counts.to_csv(COUNTS_FILE, index=False)
display(domain_counts)
print(f'Saved {COUNTS_FILE}')

,rank,domain_code,domain,observation_count,indicator_count,country_count,first_year,last_year,share_of_all_observations_pct
0,1,HSL_2,Work and job quality,1772,3,47,2010,2024,20.66
1,2,HSL_5,Health,1283,2,47,2010,2024,14.96
2,3,HSL_9,Environmental quality,1211,2,47,2010,2024,14.12
3,4,HSL_1,Income and wealth,1138,3,42,2010,2024,13.27
4,5,HSL_3,Housing,1062,2,40,2010,2024,12.38
5,6,HSL_11,Subjective well-being,929,2,47,2010,2024,10.83
6,7,HSL_7,Social connections,728,2,47,2010,2024,8.49
7,8,HSL_8,Civic engagement,247,2,47,2010,2024,2.88
8,9,HSL_6,Knowledge and skills,154,1,39,2012,2022,1.8
9,10,HSL_4,Work-life balance,51,2,16,2010,2022,0.59


Saved /Users/y2l/winter-data-challenge/data/processed/OECD_domain_counts.csv


## 12. Rank domains for Australia only

This table only includes Australian observations. The percentage column shows each domain's share of all Australian rows in the cleaned dataset.

In [12]:
australia = tidy.loc[tidy['country_code'].eq('AUS')].copy()
assert not australia.empty

au_domain_counts = (
    australia.groupby(['domain_code', 'domain'], as_index=False)
        .agg(
            observation_count=('value', 'count'),
            indicator_count=('indicator_code', 'nunique'),
            country_count=('country_code', 'nunique'),
            first_year=('year', 'min'),
            last_year=('year', 'max'),
        )
)
au_domain_counts['share_of_all_observations_pct'] = (
    au_domain_counts['observation_count'].div(len(australia)).mul(100).round(2)
)
au_domain_counts = (au_domain_counts
                    .sort_values(['observation_count', 'domain'], ascending=[False, True])
                    .reset_index(drop=True))
au_domain_counts.insert(0, 'rank', au_domain_counts.index + 1)
au_domain_counts.to_csv(AU_COUNTS_FILE, index=False)
display(au_domain_counts)
print(f'Saved {AU_COUNTS_FILE}')

,rank,domain_code,domain,observation_count,indicator_count,country_count,first_year,last_year,share_of_all_observations_pct
0,1,HSL_2,Work and job quality,45,3,1,2010,2024,24.73
1,2,HSL_5,Health,29,2,1,2010,2024,15.93
2,3,HSL_9,Environmental quality,26,2,1,2010,2024,14.29
3,4,HSL_1,Income and wealth,24,3,1,2010,2024,13.19
4,5,HSL_11,Subjective well-being,17,2,1,2010,2024,9.34
5,6,HSL_3,Housing,15,1,1,2010,2024,8.24
6,7,HSL_7,Social connections,15,1,1,2010,2024,8.24
7,8,HSL_8,Civic engagement,7,2,1,2010,2023,3.85
8,9,HSL_6,Knowledge and skills,4,1,1,2012,2022,2.2


Saved /Users/y2l/winter-data-challenge/data/processed/AU_domain_counts.csv


## TEST: Re-export and run a final check

In [13]:
tidy_check = pd.read_csv(TIDY_FILE)
summary_check = pd.read_csv(SUMMARY_FILE)
counts_check = pd.read_csv(COUNTS_FILE)
au_counts_check = pd.read_csv(AU_COUNTS_FILE)

assert len(tidy_check) == len(tidy)
assert len(summary_check) == tidy['domain'].nunique()
assert counts_check['observation_count'].is_monotonic_decreasing
assert counts_check['observation_count'].sum() == len(tidy_check)
assert set(summary_check['domain']) == set(tidy_check['domain'])
assert au_counts_check['observation_count'].is_monotonic_decreasing
assert au_counts_check['observation_count'].sum() == tidy_check['country_code'].eq('AUS').sum()
assert set(au_counts_check['domain']) == set(tidy_check.loc[tidy_check['country_code'].eq('AUS'), 'domain'])

print('All exports passed final validation:')
for path in [TIDY_FILE, SUMMARY_FILE, COUNTS_FILE, AU_COUNTS_FILE]:
    print(f'  - {path.relative_to(PROJECT_ROOT)}')

All exports passed final validation:
  - data/processed/OECD_cleaned_version.csv
  - data/processed/OECD_categories.csv
  - data/processed/OECD_domain_counts.csv
  - data/processed/AU_domain_counts.csv
